# GGUF & llama.cpp quantized inference

A refresher on **GGUF** (the file format) and **llama.cpp** (the engine that runs it): how to run
quantized LLMs efficiently on CPUs, laptops, and consumer GPUs.

**Domain:** LLM Inference, Training & Optimization  ·  **recommended addition**  ·  **runnable:** yes  ·  _cross-ref quantization (GPTQ/AWQ) and vLLM notebooks_

## 1. What & Why

**llama.cpp** is a C/C++ inference engine (by Georgi Gerganov) for running transformer LLMs with no Python
or CUDA runtime required. **GGUF** ("GPT-Generated Unified Format") is the single-file format it loads: one
binary file holding the model's metadata, tokenizer, and *quantized* weights.

**The problem it solves.** A 7B model in fp16 is ~14 GB — too big for most laptops and far too slow on CPU.
The Hugging Face / PyTorch stack also drags in gigabytes of CUDA, a Python interpreter, and a tokenizer
library. llama.cpp throws all of that out:

- **Aggressive quantization.** Weights are stored at 2–8 bits per parameter using GGUF's *k-quant* and
  *i-quant* schemes. A 7B model drops from ~14 GB (fp16) to ~4 GB (Q4_K_M) with minor quality loss, so it
  fits in RAM on a normal machine.
- **Portable, dependency-free runtime.** A single static binary (or the `llama-cpp-python` wheel) runs the
  model. It uses hand-written SIMD kernels (AVX2/AVX-512 on x86, NEON on Apple Silicon), optional Metal /
  CUDA / Vulkan offload, and **memory-maps** the GGUF so the model loads instantly and the OS pages weights
  in on demand.
- **One self-describing file.** Architecture, hyperparameters, tokenizer, chat template, and quantized
  tensors all live in the `.gguf`. No `config.json` + `tokenizer.json` + sharded `*.safetensors` to keep in
  sync — copy one file and it runs.

**Reach for it when** you want local/offline inference, CPU or Apple-Silicon deployment, edge/embedded
targets, or a tiny dependency footprint (desktop apps via Ollama/LM Studio sit on top of llama.cpp).

**Don't reach for it when** you're serving many concurrent users on datacenter GPUs (use vLLM/TGI for
throughput) or you need training/fine-tuning (llama.cpp is inference-only; train in PyTorch, then convert).

## 2. Mental Model

**GGUF is a ZIP file for a neural network; llama.cpp is the unzipper that also runs it.**

Think of three layers:

```
  ┌───────────────────────────── model.gguf (one mmap'd file) ─────────────────────────────┐
  │  HEADER     magic "GGUF" · version · tensor_count · metadata_kv_count                   │
  │  METADATA   key→value dict: arch, n_layers, n_heads, context_len, rope_theta,           │
  │             tokenizer vocab + merges, chat_template, quantization type, ...             │
  │  TENSOR     for each weight: name · shape · ggml_type (Q4_K, Q6_K, F16, ...) · offset   │
  │   INFO                                                                                  │
  │  TENSOR     the actual quantized bytes, laid out in fixed-size BLOCKS                    │
  │   DATA      (e.g. Q4_0 = 32 weights → one fp16 scale + 16 bytes of 4-bit quants)        │
  └────────────────────────────────────────────────────────────────────────────────────────┘
```

The key insight is **block quantization**: weights aren't stored one-by-one at low precision (that loses too
much). They're grouped into small blocks (32 or 256 values); each block keeps its own fp16 **scale** (and
sometimes a min), and the individual weights become tiny integers. At inference, llama.cpp dequantizes a
block back to float *just in time* inside the matmul kernel. The scale-per-block is what preserves accuracy
while still averaging ~4 bits/weight.

## 3. Key Concepts

- **GGUF** — the on-disk format. Successor to the old GGML/GGJT formats; it is *self-describing* (all
  metadata keys live in the file) and *extensible* (new keys don't break old loaders).
- **ggml** — the underlying tensor library (the autograd-free "PyTorch in C") that defines the quantized
  types and the SIMD kernels llama.cpp executes.
- **Block quantization** — weights stored in fixed-size blocks, each with its own scale (`d`) and optionally
  min (`m`). Dequant: `w ≈ d * q (+ m)`. This is why low-bit GGUF keeps quality.
- **Quant types** — the alphabet soup you pick from:
  - `F16`/`F32` — unquantized.
  - `Q4_0`, `Q5_0`, `Q8_0` — legacy "type-0" blocks (single scale per 32 weights).
  - **k-quants** `Q3_K`, `Q4_K`, `Q5_K`, `Q6_K` — superblocks of 256 with hierarchical scales; the
    `_S`/`_M`/`_L` suffix (Small/Medium/Large) mixes bit-widths across layers. **`Q4_K_M` is the default
    sweet spot.**
  - **i-quants** `IQ2_XXS`, `IQ3_XXS`, `IQ4_NL`, … — newer codebook/lattice quants for sub-4-bit with better
    quality (slower; some need an *importance matrix*).
- **imatrix (importance matrix)** — calibration stats computed from sample text; weights that matter more get
  more precision. Recommended for very low-bit quants.
- **Perplexity (PPL)** — the standard yardstick for "how much did quantization hurt?" Lower is better; you
  compare a quant's PPL against the fp16 baseline.
- **mmap** — llama.cpp memory-maps the GGUF so startup is instant and only the pages it touches load.
- **`n_gpu_layers`** — how many transformer layers to offload to GPU (Metal/CUDA/Vulkan); the rest run on CPU.
  `0` = pure CPU, `-1` = all layers on GPU.
- **`n_ctx`** — context window to allocate (sizes the KV-cache). The KV-cache itself can also be quantized.

## 4. Setup

The two runnable examples below are **pure Python + NumPy** — they construct and parse a real GGUF file and
simulate the Q4_0 block-quantization math, so they run anywhere with no model download.

To actually run a quantized model, install the Python bindings:

```bash
pip install llama-cpp-python          # CPU build (uses your CPU's SIMD)
# Apple Silicon Metal offload:
CMAKE_ARGS="-DGGML_METAL=on"   pip install llama-cpp-python
# NVIDIA CUDA offload:
CMAKE_ARGS="-DGGML_CUDA=on"    pip install llama-cpp-python
```

Or use the C++ binary / tooling directly:

```bash
git clone https://github.com/ggml-org/llama.cpp && cd llama.cpp && cmake -B build && cmake --build build -j
# convert HF weights -> GGUF, then quantize:
python convert_hf_to_gguf.py /path/to/hf-model --outfile model-f16.gguf
./build/bin/llama-quantize model-f16.gguf model-Q4_K_M.gguf Q4_K_M
./build/bin/llama-cli -m model-Q4_K_M.gguf -p "Hello" -n 64
```

The real-inference cell at the end is **gated behind `os.getenv` + an import check**, so this notebook
executes top-to-bottom even without `llama-cpp-python` or a model file.

In [1]:
import os, sys, struct
import numpy as np

# Are the real bindings importable here?
try:
    import llama_cpp  # noqa: F401
    LLAMA_CPP_AVAILABLE = True
    llama_cpp_version = llama_cpp.__version__
except Exception as e:
    LLAMA_CPP_AVAILABLE = False
    llama_cpp_version = f"not installed ({type(e).__name__})"

RUN_LLAMA = bool(os.getenv("RUN_LLAMA_CPP"))  # gate the heavy download/inference cell

print(f"python            : {sys.version.split()[0]}")
print(f"numpy             : {np.__version__}")
print(f"llama-cpp-python  : {llama_cpp_version}")
print(f"RUN_LLAMA_CPP gate: {RUN_LLAMA}")
print("Real inference will run." if (LLAMA_CPP_AVAILABLE and RUN_LLAMA)
      else "Real inference is skipped; the pure-Python examples run regardless.")

python            : 3.13.7
numpy             : 2.5.0
llama-cpp-python  : not installed (ModuleNotFoundError)
RUN_LLAMA_CPP gate: False
Real inference is skipped; the pure-Python examples run regardless.


## 5. Worked Examples

Three examples, "always runs" → "needs a model":

1. **Write & parse a real GGUF file** — build a tiny valid `.gguf` byte-for-byte and read its metadata back,
   so the format stops being a black box (pure Python `struct`).
2. **Q4_0 block quantization** — implement llama.cpp's simplest quant and measure the size/quality trade-off
   (pure NumPy).
3. **Run a quantized model** with `llama-cpp-python` (gated download + inference).

### Example 1 — Write and parse a real GGUF file

GGUF's header is dead simple: the ASCII magic `GGUF`, a `uint32` version, a `uint64` tensor count, a `uint64`
metadata-KV count, then the metadata entries. Each entry is `key (string)`, a `uint32` value-type tag, then
the value. Strings are a `uint64` length followed by UTF-8 bytes. We'll write a header with zero tensors and
a few metadata keys, then parse it back — exactly what a loader does to discover a model's architecture.

In [2]:
# GGUF value-type tags we use (see the spec for the full enum)
GGUF_U32, GGUF_F32, GGUF_STR = 4, 6, 8
GGUF_MAGIC, GGUF_VERSION = b"GGUF", 3

def _u32(x):  return struct.pack("<I", x)
def _u64(x):  return struct.pack("<Q", x)
def _f32(x):  return struct.pack("<f", x)
def _str(s):  b = s.encode("utf-8"); return _u64(len(b)) + b   # uint64 length + bytes

def write_gguf_header(metadata: dict) -> bytes:
    """Serialize a GGUF file with metadata and zero tensors."""
    out = bytearray()
    out += GGUF_MAGIC + _u32(GGUF_VERSION)
    out += _u64(0)                 # tensor_count
    out += _u64(len(metadata))     # metadata_kv_count
    for key, (vtype, value) in metadata.items():
        out += _str(key) + _u32(vtype)
        if   vtype == GGUF_STR: out += _str(value)
        elif vtype == GGUF_U32: out += _u32(value)
        elif vtype == GGUF_F32: out += _f32(value)
    return bytes(out)

blob = write_gguf_header({
    "general.architecture":      (GGUF_STR, "llama"),
    "general.name":              (GGUF_STR, "tiny-demo"),
    "llama.context_length":      (GGUF_U32, 4096),
    "llama.block_count":         (GGUF_U32, 22),
    "llama.attention.head_count":(GGUF_U32, 32),
    "llama.rope.freq_base":      (GGUF_F32, 10000.0),
    "general.file_type":         (GGUF_U32, 15),   # 15 = Q4_K_M
})
print(f"wrote {len(blob)} bytes; first 4 = {blob[:4]!r}")

wrote 290 bytes; first 4 = b'GGUF'


In [3]:
# Now parse it back, the way a loader discovers the model's config.
class Reader:
    def __init__(self, buf): self.buf, self.pos = buf, 0
    def take(self, n):
        b = self.buf[self.pos:self.pos + n]; self.pos += n; return b
    def u32(self): return struct.unpack("<I", self.take(4))[0]
    def u64(self): return struct.unpack("<Q", self.take(8))[0]
    def f32(self): return struct.unpack("<f", self.take(4))[0]
    def string(self): return self.take(self.u64()).decode("utf-8")

def parse_gguf_header(buf: bytes) -> dict:
    r = Reader(buf)
    assert r.take(4) == GGUF_MAGIC, "not a GGUF file"
    version, n_tensors, n_kv = r.u32(), r.u64(), r.u64()
    meta = {}
    for _ in range(n_kv):
        key, vtype = r.string(), r.u32()
        meta[key] = {GGUF_STR: r.string, GGUF_U32: r.u32, GGUF_F32: r.f32}[vtype]()
    return {"version": version, "n_tensors": n_tensors, "metadata": meta}

info = parse_gguf_header(blob)
print(f"GGUF v{info['version']}, {info['n_tensors']} tensors, {len(info['metadata'])} metadata keys\n")
for k, v in info["metadata"].items():
    print(f"  {k:32s} = {v}")

GGUF v3, 0 tensors, 7 metadata keys

  general.architecture             = llama
  general.name                     = tiny-demo
  llama.context_length             = 4096
  llama.block_count                = 22
  llama.attention.head_count       = 32
  llama.rope.freq_base             = 10000.0
  general.file_type                = 15


Round-tripped cleanly — the file *is* its own config. A real loader continues past the metadata into the
tensor-info section (name, shape, `ggml_type`, byte offset for each weight) and then mmaps the tensor data.
Notice `general.file_type = 15`, which is how the file announces it's a `Q4_K_M` model: the loader knows to
use the Q4_K dequant kernel before it reads a single weight.

### Example 2 — Q4_0 block quantization, end to end

This is llama.cpp's simplest quant, and it shows the whole idea. A **block** is 32 consecutive weights; we
store one fp16 **scale** `d` plus 32 **4-bit** integers. Dequant is `w ≈ d * (q - 8)`. Let's quantize a real
weight vector, dequantize it, and measure the error and the compression ratio.

In [4]:
QK = 32  # Q4_0 block size

def quantize_q4_0(w: np.ndarray):
    """Quantize a 1-D float array in blocks of 32 -> (scales fp16, 4-bit quants)."""
    w = w.reshape(-1, QK).astype(np.float32)
    amax_idx = np.argmax(np.abs(w), axis=1)
    amax = w[np.arange(w.shape[0]), amax_idx]      # signed value of largest magnitude
    d = amax / -8.0                                # scale (fp16 in real GGUF)
    d_safe = np.where(d == 0, 1.0, d)
    q = np.clip(np.round(w / d_safe[:, None]) + 8, 0, 15).astype(np.uint8)
    return d.astype(np.float16), q

def dequantize_q4_0(d: np.ndarray, q: np.ndarray) -> np.ndarray:
    return (d.astype(np.float32)[:, None] * (q.astype(np.float32) - 8)).reshape(-1)

rng = np.random.default_rng(0)
weights = rng.standard_normal(4096).astype(np.float32)   # one weight row

d, q = quantize_q4_0(weights)
recon = dequantize_q4_0(d, q)

rms   = np.sqrt(np.mean((weights - recon) ** 2))
n     = weights.size
fp16_bytes = n * 2
# Q4_0: per 32-weight block -> 2 bytes (fp16 scale) + 16 bytes (32 * 4 bits)
q4_bytes   = (n // QK) * (2 + QK // 2)

print(f"weights              : {n} values")
print(f"fp16 size            : {fp16_bytes:>6d} bytes   (16.0 bits/weight)")
print(f"Q4_0 size            : {q4_bytes:>6d} bytes   ({q4_bytes * 8 / n:.2f} bits/weight)")
print(f"compression          : {fp16_bytes / q4_bytes:.2f}x smaller")
print(f"RMS error            : {rms:.4f}")
print(f"max abs error        : {np.abs(weights - recon).max():.4f}")
print(f"corr(orig, recon)    : {np.corrcoef(weights, recon)[0, 1]:.4f}")

weights              : 4096 values
fp16 size            :   8192 bytes   (16.0 bits/weight)
Q4_0 size            :   2304 bytes   (4.50 bits/weight)
compression          : 3.56x smaller
RMS error            : 0.0840
max abs error        : 0.2955
corr(orig, recon)    : 0.9965


~3.6× smaller at 4.5 bits/weight, and the reconstruction still correlates ~0.99 with the original — that's
the whole pitch of GGUF quantization. The error comes entirely from squeezing 32 floats through a single
shared scale; real **k-quants** (`Q4_K`) cut it further by adding a second level of per-sub-block scales and
a block *min*, and **i-quants** do better still with codebooks. The principle is identical — only the
bookkeeping per block changes.

### Example 3 — Run a quantized model (gated)

The canonical way to run a GGUF in Python. It's **gated** behind `RUN_LLAMA_CPP=1` *and* an import check so
this notebook executes without the dependency or a multi-GB download. `from_pretrained` pulls a single GGUF
file straight from the Hugging Face Hub.

In [5]:
if LLAMA_CPP_AVAILABLE and RUN_LLAMA:
    from llama_cpp import Llama

    # Pulls one ~1 GB GGUF file from the Hub and runs it on CPU.
    llm = Llama.from_pretrained(
        repo_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
        filename="*Q4_K_M.gguf",     # pick the Q4_K_M quant
        n_ctx=2048,
        n_gpu_layers=0,              # 0 = pure CPU; -1 = offload all layers to GPU
        verbose=False,
    )
    out = llm.create_chat_completion(
        messages=[{"role": "user", "content": "In one sentence, what is GGUF?"}],
        max_tokens=64, temperature=0.7, seed=0,
    )
    print(out["choices"][0]["message"]["content"].strip())
else:
    # Show the exact call shape without downloading anything.
    print("[skipped: pip install llama-cpp-python && set RUN_LLAMA_CPP=1 to run for real]")
    print("Call shape:")
    print('  llm = Llama.from_pretrained(repo_id="...-GGUF", filename="*Q4_K_M.gguf",')
    print('                              n_ctx=2048, n_gpu_layers=0)')
    print('  llm.create_chat_completion(messages=[{"role": "user", "content": "..."}])')
    print('  # or low-level:  llm("prompt", max_tokens=64)["choices"][0]["text"]')

[skipped: pip install llama-cpp-python && set RUN_LLAMA_CPP=1 to run for real]
Call shape:
  llm = Llama.from_pretrained(repo_id="...-GGUF", filename="*Q4_K_M.gguf",
                              n_ctx=2048, n_gpu_layers=0)
  llm.create_chat_completion(messages=[{"role": "user", "content": "..."}])
  # or low-level:  llm("prompt", max_tokens=64)["choices"][0]["text"]


## 6. Gotchas & Pitfalls

- **Picking the wrong quant.** `Q4_K_M` is the default sweet spot (~4.5 bpw, tiny quality loss). Below
  `Q3_K` quality degrades fast — prefer an **i-quant** (`IQ3_XXS`/`IQ2`) with an **imatrix** over a plain
  `Q3_K_S` if you must go that low. `Q8_0` is near-lossless but barely smaller than fp16; usually pointless.
- **`Q4_0` vs `Q4_K_M` confusion.** The legacy `_0`/`_1` types are simpler and a bit faster but lower
  quality than the `_K` k-quants at the same bit width. Default to k-quants unless you have a reason.
- **Wrong / missing chat template.** If output is rambling or ignores the system prompt, you're likely
  feeding raw text to a chat model. Use `create_chat_completion` (it applies the GGUF's embedded chat
  template) instead of the bare `__call__` completion API.
- **`n_ctx` and the KV-cache eat RAM.** Asking for `n_ctx=32768` allocates a big KV-cache up front and can
  OOM even though the weights fit. Size it to what you need; consider KV-cache quantization (`type_k`/`type_v`).
- **Stale/incompatible GGUF.** The format has changed (GGML → GGUF, and metadata keys evolve). A very old
  `.gguf` or a mismatched llama.cpp version can fail to load — re-download or re-convert with current tooling.
- **CPU build with no SIMD / wrong threads.** A generic wheel may miss AVX-512 or set bad thread counts.
  Tune `n_threads` to physical cores; build with the right `CMAKE_ARGS` for Metal/CUDA to actually use the GPU.
- **Quantize from F16, not from another quant.** Always convert HF → F16 GGUF first, then quantize. Quantizing
  an already-quantized file compounds error.
- **`from_pretrained` filename globs.** A repo holds many quants; an ambiguous `filename` glob matches several
  files and errors. Be specific (`"*Q4_K_M.gguf"`).

## 7. When to Use vs Alternatives

| Tool / format | Best at | Trade-off vs llama.cpp / GGUF |
|---|---|---|
| **llama.cpp + GGUF** | CPU / Apple Silicon / edge / local desktop; tiny footprint; one-file models; broad quant menu | Low *concurrent* throughput; inference-only; not for big GPU fleets |
| **Ollama / LM Studio** | Friendly local UX on top of llama.cpp (pulls, model mgmt, server) | Less low-level control; still single-machine; it *is* llama.cpp underneath |
| **vLLM / TGI** | High-throughput multi-tenant GPU serving (PagedAttention, continuous batching) | GPU-only, heavy stack; overkill on a laptop. See the **vLLM** notebook |
| **GPTQ / AWQ (+ ExLlama)** | 4-bit on NVIDIA GPUs with fast kernels | GPU-centric; weaker CPU story than GGUF. See the **quantization (GPTQ/AWQ)** notebook |
| **bitsandbytes (HF)** | Quick 4/8-bit load inside the Transformers training/inference stack | Bigger deps; slower than llama.cpp on CPU; great for QLoRA fine-tuning |
| **MLX (Apple)** | Native Apple-Silicon training + inference | Apple-only; smaller ecosystem than GGUF |
| **Hosted APIs** | Zero ops, frontier models | No local control; cost at scale; data leaves your machine |

**Rule of thumb:** running locally on a laptop, CPU, or Mac, or shipping a model inside a desktop/edge app →
**GGUF + llama.cpp** (often via Ollama). Serving many users on datacenter GPUs → **vLLM/TGI**. 4-bit on a
single NVIDIA GPU for max speed → **GPTQ/AWQ**. Fine-tuning → train in PyTorch (QLoRA/Unsloth), then convert
the result to GGUF to ship it.

## 8. Resources

- **llama.cpp repo** — https://github.com/ggml-org/llama.cpp (build, tools, `llama-quantize`, examples)
- **GGUF format spec** — https://github.com/ggml-org/ggml/blob/master/docs/gguf.md (the byte layout used above)
- **llama-cpp-python docs** — https://llama-cpp-python.readthedocs.io/ (the Python bindings & OpenAI-style API)
- **k-quants PR / discussion** — https://github.com/ggml-org/llama.cpp/pull/1684 (how Q*_K superblocks work)
- **i-quants / imatrix overview** — https://github.com/ggml-org/llama.cpp/pull/4773 (sub-4-bit quantization)
- **Ollama** — https://github.com/ollama/ollama (the popular llama.cpp-based local runner)
- **Hugging Face GGUF docs** — https://huggingface.co/docs/hub/gguf (finding & using GGUF models on the Hub)